### Departamento: Análisis de Operaciones y Gestión de Inventario
¿Cuál es la disponibilidad media de los alojamientos
turísticos en los distintos plazos (30, 60, 90 y 365 días) en cada ciudad?

In [28]:
import pandas as pd
import os
import matplotlib.pyplot as plt


In [19]:
# Cargar datos
ruta = os.path.join('..', 'Data', 'TouristAccommodationClean19012026.csv')
df = pd.read_csv(ruta)

In [20]:
availability_cols = ["availability_30", "availability_60", "availability_90", "availability_365"]

# Filtrar solo alojamientos con disponibilidad
df_available = df[df["has_availability"] == True]

# Calcular disponibilidad media por ciudad
availability_by_city = (
    df_available
    .groupby("city")[availability_cols]
    .mean()
    .reset_index()
)

# Ordenar por disponibilidad a 30 días (corto plazo) para negocio
availability_by_city = availability_by_city.sort_values(by="availability_30").round(2)

# Renombrar el nombre de las columnas
new_column_names = {
    "city": "ciudad",
    "availability_30": "disponibilidad_30",
    "availability_60": "disponibilidad_60",
    "availability_90": "disponibilidad_90",
    "availability_365": "disponibilidad_365"
}

availability_by_city.rename(columns=new_column_names, inplace=True)

availability_by_city

,ciudad,disponibilidad_30,disponibilidad_60,disponibilidad_90,disponibilidad_365
2,madrid,10.20,23.67,39.07,158.78
0,barcelona,11.33,26.07,43.09,180.79
3,malaga,12.21,28.60,47.41,203.09
4,mallorca,13.32,28.55,45.06,210.87
7,valencia,13.52,29.64,47.71,185.24
6,sevilla,13.84,30.52,49.49,201.55
1,girona,14.61,31.35,48.42,195.39
5,menorca,14.97,30.89,47.83,200.50


**Disponibilidad media de los alojamientos por ciudad y plazo**

In [105]:
import plotly.express as px

disponibilidad_plazos = ['disponibilidad_30', 'disponibilidad_60', 'disponibilidad_90', 'disponibilidad_365']

fig = px.bar(
    availability_by_city,
    x='ciudad',                   
    y=disponibilidad_plazos,  
    barmode='group',
    title="Disponibilidad media de los alojamientos por ciudad y plazo",
    color_discrete_sequence=["#9cbcec", "#499dd4", "#1A64A1", "#052041"]  
)

fig.update_traces(
    texttemplate='<b>%{y:.0f}</b>', 
    textposition='outside',  # 👈 Bar'ın dışında göster
    textfont_size=9
)

fig.update_layout(
    
    title={
        'text': "Disponibilidad Media (Días) por Ciudad ",
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size':20, 'family':'Arial', 'color':'black', 'weight':'bold'}
    },

    xaxis_title='',
    yaxis_title='',

    xaxis_tickangle=-45,
    xaxis_tickfont={'size':12, 'family':'Arial', 'color':'black', 'weight':'bold'},
    yaxis_tickfont={'size':12, 'family':'Arial', 'color':'black', 'weight':'bold'},

   
    legend_title_text="Plazo",
    template="plotly_white"
)
fig.update_layout(
    bargap=0.25,       
    bargroupgap=0.05    
)


fig.update_traces(width=0.22)
fig.update_layout(
    yaxis_showticklabels=False
)

fig.show()


**Ocupación media del ocupacion_corto_plazo y ocupacion_largo_plazo por ciudad**


In [ ]:
# Definimos los horizontes temporales
plazos = [30, 60, 90, 365]

# Cálculo de la tasa de ocupación por plazo
for p in plazos:
    df[f"tasa_ocupacion_{p}"] = ((p - df[f"availability_{p}"]) / p) * 100
    df[f"tasa_ocupacion_{p}"] = df[f"tasa_ocupacion_{p}"].clip(0, 100)

# Índice de ocupacion a corto plazo
df["ocupacion_corto_plazo"] = (
    df["tasa_ocupacion_30"] * 0.6 +
    df["tasa_ocupacion_60"] * 0.4
)

# Índice de ocupacion a largo plazo
df["ocupacion_largo_plazo"] = (
    df["tasa_ocupacion_90"] * 0.4 +
    df["tasa_ocupacion_365"] * 0.6
)

# KPIs promedio por ciudad
kpis_ciudad = df.groupby("city")[[
    "ocupacion_corto_plazo",
    "ocupacion_largo_plazo"
]].mean().reset_index()

kpis_ciudad

,city,ocupacion_corto_plazo,ocupacion_largo_plazo
0,barcelona,60.784203,50.890027
1,girona,49.873778,46.360321
2,madrid,63.272015,55.713539
3,malaga,56.516190,45.546371
4,mallorca,53.738345,44.868974
5,menorca,49.465721,45.783996
6,sevilla,51.972299,44.873373
7,valencia,53.200869,48.346378


In [59]:
mediana_corto_plazo = kpis_ciudad["ocupacion_corto_plazo"].median()
mediana_largo_plazo = kpis_ciudad["ocupacion_largo_plazo"].median()

In [107]:
import plotly.graph_objects as go

fig = go.Figure()


fig.add_trace(
    go.Bar(
        x=kpis_ciudad["city"],
        y=kpis_ciudad["ocupacion_corto_plazo"],
        marker_color="#43A7D2",        
        name="Ocupacion corto plazo"
    )
)

# La linea de median
fig.add_hline(
    y=mediana_corto_plazo,
    line_dash="dash",
    line_color="black",
    line_width=1.5,
    annotation_text="Mediana",
    annotation_position="top right"
)

fig.update_layout(
    title={
        'text':"Ocupación de Corto Plazo (30-60) por Ciudad",
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size':20, 'family':'Arial', 'color':'black', 'weight':'bold'}
    },
    xaxis_title='',
    xaxis_title_font={'size':16, 'family':'Arial', 'color':'black', 'weight':'bold'},
    yaxis_title_font={'size':16, 'family':'Arial', 'color':'black', 'weight':'bold'},

    xaxis_tickfont={'size':12, 'family':'Arial', 'color':'black', 'weight':'bold'},
    yaxis_tickfont={'size':12, 'family':'Arial', 'color':'black', 'weight':'bold'},
    
    
    yaxis_title="<b>Ocupación de corto plazo (%)</b>",
    bargap=0.2,
    template="simple_white"
)

fig.show()

In [108]:
import plotly.graph_objects as go

fig = go.Figure()


fig.add_trace(
    go.Bar(
        x=kpis_ciudad["city"],
        y=kpis_ciudad["ocupacion_largo_plazo"],
        marker_color="#1D2645",
        name="Ocupacion corto plazo"
    )
)

# La linea de median
fig.add_hline(
    y=mediana_corto_plazo,
    line_dash="dash",
    line_color="black",
    line_width=1.5,
    annotation_text="Mediana",
    annotation_position="top right"
)

fig.update_layout(
    title={
        'text':"Ocupación de Largo Plazo (90-365) por Ciudad",
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size':20, 'family':'Arial', 'color':'black', 'weight':'bold'}
    },
    xaxis_title='',
    
    xaxis_title_font={'size':16, 'family':'Arial', 'color':'black', 'weight':'bold'},
    yaxis_title_font={'size':16, 'family':'Arial', 'color':'black', 'weight':'bold'},

    xaxis_tickfont={'size':12, 'family':'Arial', 'color':'black', 'weight':'bold'},
    yaxis_tickfont={'size':12, 'family':'Arial', 'color':'black', 'weight':'bold'},
    
    
    yaxis_title="<b>Ocupación de largo plazo (%)</b>",
    bargap=0.2,
    template="simple_white"
)

fig.show()